In [1]:
import pandas as pd

In [2]:
# load the training and test datasets
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

#### Dataset Overview
Inspecting the size, structure, columns and data types of the training and test datasets.

In [3]:
# Inspect the basic structure of both datasets
print("Training data shape:", train.shape)
print("Test data shape:", test.shape)

print("\nFirst five training rows:")
display(train.head())

print("\nTRAIN DATA:")
train.info()

print("\nTEST DATA:")
test.info()

Training data shape: (103904, 25)
Test data shape: (25976, 25)

First five training rows:


,Unnamed: 0,id,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,...,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction
0,0,70172,Male,Loyal Customer,13,Personal Travel,Eco Plus,460,3,4,...,5,4,3,4,4,5,5,25,18.0,neutral or dissatisfied
1,1,5047,Male,disloyal Customer,25,Business travel,Business,235,3,2,...,1,1,5,3,1,4,1,1,6.0,neutral or dissatisfied
2,2,110028,Female,Loyal Customer,26,Business travel,Business,1142,2,2,...,5,4,3,4,4,4,5,0,0.0,satisfied
3,3,24026,Female,Loyal Customer,25,Business travel,Business,562,2,5,...,2,2,5,3,1,4,2,11,9.0,neutral or dissatisfied
4,4,119299,Male,Loyal Customer,61,Business travel,Business,214,3,3,...,3,3,4,4,3,3,3,0,0.0,satisfied



TRAIN DATA:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103904 entries, 0 to 103903
Data columns (total 25 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   Unnamed: 0                         103904 non-null  int64  
 1   id                                 103904 non-null  int64  
 2   Gender                             103904 non-null  object 
 3   Customer Type                      103904 non-null  object 
 4   Age                                103904 non-null  int64  
 5   Type of Travel                     103904 non-null  object 
 6   Class                              103904 non-null  object 
 7   Flight Distance                    103904 non-null  int64  
 8   Inflight wifi service              103904 non-null  int64  
 9   Departure/Arrival time convenient  103904 non-null  int64  
 10  Ease of Online booking             103904 non-null  int64  
 11  Gate location             

#### Data Quality Inspection
Checking for missing and unknown values, duplicates, categories and class distribution.

In [4]:
# Check missing values, duplicates, categories, numerical ranges and target distribution

print("=== MISSING VALUES ===")
print("\nTrain:")
print(train.isnull().sum())

print("\nTest:")
print(test.isnull().sum())

print("\n=== DUPLICATES ===")
print("Train duplicate rows:", train.duplicated().sum())
print("Test duplicate rows:", test.duplicated().sum())

print("\n=== CATEGORICAL VALUES ===")
categorical_columns = ["Gender", "Customer Type", "Type of Travel", "Class", "satisfaction"]

for column in categorical_columns:
    print(f"{column}: {train[column].unique()}")

print("\n=== NUMERICAL RANGES ===")
print(train.select_dtypes(include="number").agg(["min", "max"]))

print("\n=== TARGET DISTRIBUTION ===")
print(train["satisfaction"].value_counts())
print("\nPercentages:")
print((train["satisfaction"].value_counts(normalize=True) * 100).round(2))

=== MISSING VALUES ===

Train:
Unnamed: 0                             0
id                                     0
Gender                                 0
Customer Type                          0
Age                                    0
Type of Travel                         0
Class                                  0
Flight Distance                        0
Inflight wifi service                  0
Departure/Arrival time convenient      0
Ease of Online booking                 0
Gate location                          0
Food and drink                         0
Online boarding                        0
Seat comfort                           0
Inflight entertainment                 0
On-board service                       0
Leg room service                       0
Baggage handling                       0
Checkin service                        0
Inflight service                       0
Cleanliness                            0
Departure Delay in Minutes             0
Arrival Delay in Minutes  

#### Train-Test Consistency and Leakage Checks
Checking whether the train and test datasets are separated correctly and identifying variables that could cause data leakage.

In [5]:
# Check train-test consistency, shared IDs and possible identifier leakage

print("Same columns:", train.columns.equals(test.columns))

shared_ids = set(train["id"]) & set(test["id"])
print("Shared IDs:", len(shared_ids))

print("Unique train IDs:", train["id"].nunique(), "of", len(train))
print("Unique test IDs:", test["id"].nunique(), "of", len(test))

print("\nIdentifier columns:")
print(train[["Unnamed: 0", "id"]].head())

Same columns: True
Shared IDs: 0
Unique train IDs: 103904 of 103904
Unique test IDs: 25976 of 25976

Identifier columns:
   Unnamed: 0      id
0           0   70172
1           1    5047
2           2  110028
3           3   24026
4           4  119299


#### Feature Relationships
Checking correlations between numerical features to identify strongly related variables.

In [6]:
# Find strongly correlated numerical feature pairs
correlation = train.select_dtypes(include="number").corr()

strong_pairs = (
    correlation.where(~correlation.eq(1))
    .stack()
    .sort_values(ascending=False)
)

print(strong_pairs[strong_pairs > 0.7].drop_duplicates())

Arrival Delay in Minutes  Departure Delay in Minutes    0.965481
Inflight wifi service     Ease of Online booking        0.715856
dtype: float64
